# Unit06_Kernel_PCA | 副標題：化工製程的非線性降維分析

本 Notebook 以程式生成的化工反應器模擬數據，建立 **Kernel PCA (核主成分分析)** 模型來進行非線性降維分析與製程監控。

## 目標
- 理解 Kernel PCA 的核心原理與數學基礎
- 掌握核函數的選擇與超參數調整方法
- 學會使用 scikit-learn 實作 Kernel PCA 模型
- 比較線性 PCA 與 Kernel PCA 的效果差異（含定量分離指標）
- 應用 Kernel PCA 建立非線性製程監控統計量（T² 與 SPE）

資料背景：模擬化工反應器的非線性操作數據，包含溫度、壓力、流量、濃度等製程變數（數據於 Notebook 內生成，不需下載外部檔案）

---
## 0. 環境設定

In [1]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'P2_Unit06_Kernel_PCA'
SOURCE_DATA_DIR = 'reactor_operation'

try:
  from google.colab import drive
  IN_COLAB = True
  print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
  drive.mount('/content/drive', force_remount=True)
except ImportError:
  IN_COLAB = False
  print("✓ 偵測到 Local 環境")

try:
  shortcut_path = '/content/ChemE-3590'
  os.remove(shortcut_path)
except FileNotFoundError:
  pass

if IN_COLAB:
  source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3590')
  os.symlink(source_path, shortcut_path)
  shortcut_path = Path(shortcut_path)
  if source_path.exists():
    NOTEBOOK_DIR = shortcut_path / 'Part_2' / 'Unit06'
    DATA_DIR = NOTEBOOK_DIR / 'data' / SOURCE_DATA_DIR
    OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    print(f"✓ Notebook 目錄: {NOTEBOOK_DIR}")
    print(f"✓ 數據目錄: {DATA_DIR}")
    print(f"✓ 輸出目錄: {OUTPUT_DIR}")
  else:
    raise FileNotFoundError(f"找不到 Google Drive 中的課程資料夾：{source_path}")
else:
  NOTEBOOK_DIR = Path.cwd()
  DATA_DIR = NOTEBOOK_DIR / 'data' / SOURCE_DATA_DIR
  OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
  print(f"✓ Notebook 目錄: {NOTEBOOK_DIR}")
  print(f"✓ 數據目錄: {DATA_DIR}")
  print(f"✓ 輸出目錄: {OUTPUT_DIR}")

# ========================================
# 建立輸出資料夾
# ========================================
FIG_DIR = OUTPUT_DIR / 'figs'
MODEL_DIR = OUTPUT_DIR / 'models'

for dir_path in [OUTPUT_DIR, FIG_DIR, MODEL_DIR]:
  dir_path.mkdir(parents=True, exist_ok=True)

print(f"\n✓ 圖片輸出目錄: {FIG_DIR}")
print(f"✓ 模型輸出目錄: {MODEL_DIR}")

✓ 偵測到 Local 環境
✓ Notebook 目錄: D:\MyGit\ChemE-3590\Part_2\Unit06
✓ 數據目錄: D:\MyGit\ChemE-3590\Part_2\Unit06\data\reactor_operation
✓ 輸出目錄: D:\MyGit\ChemE-3590\Part_2\Unit06\outputs\P2_Unit06_Kernel_PCA

✓ 圖片輸出目錄: D:\MyGit\ChemE-3590\Part_2\Unit06\outputs\P2_Unit06_Kernel_PCA\figs
✓ 模型輸出目錄: D:\MyGit\ChemE-3590\Part_2\Unit06\outputs\P2_Unit06_Kernel_PCA\models


---
## 1. 載入相關套件

In [2]:
# 數據處理
import numpy as np
import pandas as pd

# 視覺化
import matplotlib.pyplot as plt
import seaborn as sns

# 機器學習
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler, KernelCenterer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import pairwise_kernels, rbf_kernel
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

# 模型儲存
import joblib

# 警告訊息處理
import warnings
warnings.filterwarnings('ignore')

# 設定隨機種子
SEED = 42
np.random.seed(SEED)

# 設定圖表風格
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['axes.unicode_minus'] = False

print("✓ 所有套件載入完成")

✓ 所有套件載入完成


---
## 2. 生成模擬數據

**情境說明**：
模擬化工反應器的**非線性操作數據**，特別設計能展示 Kernel PCA 優勢的數據結構。數據由兩個潛在座標 $(x_1, x_2)$ 生成，再透過非線性公式轉換為 6 個製程變數：

- **模式 A (Bimodal - 雙峰分佈)**：
  - 內環操作（徑向距離 $r \approx 2$）：低溫低壓穩態
  - 外環操作（徑向距離 $r \approx 5$）：高溫高壓動態
  - 代表反應器在兩種極端條件下的操作模式

- **模式 B (Middle - 中間環)**：
  - 中環操作（徑向距離 $r \approx 3.5$）：中等溫壓過渡態
  - 代表反應器在中等條件下的穩定操作

**關鍵挑戰（為何傳統 PCA 效果受限）**：
- 模式 B 的徑向距離介於模式 A 的內環與外環之間，模式 A 的兩群被模式 B「夾在中間」
- 因此**沒有任何一個線性方向（超平面）能把模式 A 與模式 B 完整分開**（線性不可分）
- 線性 PCA 只能保留變異最大的方向，無法把「徑向距離」這個非線性特徵展開
- 非線性核函數（如 RBF）以樣本間距離為基礎，較能反映這種環狀的「內-中-外」層次結構

**注意**：本案例的兩模式在 PCA 平面上並非完全重疊，而是呈現「被夾擊、線性不可分」的結構。因此本 Notebook 同時使用視覺化與定量指標（silhouette 係數、kNN 交叉驗證準確率）來比較降維效果，而非僅憑目視判斷。

**數據包含以下製程變數**：
- 反應器溫度 (Temperature, °C)
- 反應器壓力 (Pressure, bar)
- 進料流量 (Feed Flow Rate, kg/h)
- 反應物A濃度 (Concentration A, mol/L)
- 反應物B濃度 (Concentration B, mol/L)
- 產物濃度 (Product Concentration, mol/L)

所有變數皆為徑向距離與角度的非線性函數。數據為純數值模擬，變數範圍（例如濃度 B 可能出現負值）僅供示範，不代表真實物理量。

In [3]:
def generate_nonlinear_reactor_data(n_samples=400, noise_level=0.15, random_seed=42):
    """
    生成具有非線性關係的反應器數據（內外雙環 + 中環）

    - 模式 A：內環 (radius ≈ 2) + 外環 (radius ≈ 5)
    - 模式 B：中環 (radius ≈ 3.5)，被模式 A 夾在中間，因此與 A 線性不可分

    Parameters
    ----------
    n_samples : int
        樣本數量
    noise_level : float
        噪音強度
    random_seed : int
        隨機種子

    Returns
    -------
    df : pd.DataFrame
        生成的數據
    """
    np.random.seed(random_seed)
    
    # ========================================
    # 模式 A：內環 + 外環（雙峰分佈）
    # ========================================
    n_mode_a = n_samples // 2
    n_inner = n_mode_a // 2  # 一半在內環
    n_outer = n_mode_a - n_inner  # 一半在外環
    
    # 內環部分 (radius ≈ 2.0)
    theta_inner = np.linspace(0, 2*np.pi, n_inner, endpoint=False) + noise_level * 2 * np.random.randn(n_inner)
    radius_inner = 2.0 + noise_level * np.random.randn(n_inner)
    x1_inner = radius_inner * np.cos(theta_inner)
    x2_inner = radius_inner * np.sin(theta_inner)
    
    # 外環部分 (radius ≈ 5.0)
    theta_outer = np.linspace(0, 2*np.pi, n_outer, endpoint=False) + noise_level * 2 * np.random.randn(n_outer)
    radius_outer = 5.0 + noise_level * np.random.randn(n_outer)
    x1_outer = radius_outer * np.cos(theta_outer)
    x2_outer = radius_outer * np.sin(theta_outer)
    
    # 合併模式 A 的內外環
    x1_a = np.concatenate([x1_inner, x1_outer])
    x2_a = np.concatenate([x2_inner, x2_outer])
    
    # ========================================
    # 模式 B：中環（單峰分佈）
    # ========================================
    n_mode_b = n_samples - n_mode_a
    theta_b = np.linspace(0, 2*np.pi, n_mode_b, endpoint=False) + noise_level * 2 * np.random.randn(n_mode_b)
    radius_b = 3.5 + noise_level * np.random.randn(n_mode_b)
    x1_b = radius_b * np.cos(theta_b)
    x2_b = radius_b * np.sin(theta_b)
    
    # ========================================
    # 合併所有數據
    # ========================================
    x1 = np.concatenate([x1_a, x1_b])
    x2 = np.concatenate([x2_a, x2_b])
    radial_distance = np.sqrt(x1**2 + x2**2)
    angle = np.arctan2(x2, x1)
    
    # ========================================
    # 基於位置生成化工變數
    # ========================================
    
    # 溫度：與徑向距離和角度的非線性關係
    temperature = 320 + 15 * radial_distance + 8 * np.sin(2*angle) * radial_distance + noise_level * 10 * np.random.randn(n_samples)
    
    # 壓力：非線性函數
    pressure = 10 + 0.3 * radial_distance**2 - 2 * radial_distance * np.cos(angle) + noise_level * 0.8 * np.random.randn(n_samples)
    
    # 進料流量：角度和徑向的交互作用
    feed_flow = 1000 + 40 * radial_distance * (1 + 0.3*np.cos(3*angle)) + noise_level * 35 * np.random.randn(n_samples)
    
    # 濃度 A：徑向二次函數
    conc_a = 8 + 0.6 * radial_distance**2 + noise_level * 1.5 * np.random.randn(n_samples)
    
    # 濃度 B：非線性函數
    conc_b = 6 + 0.4 * radial_distance**2 * np.cos(angle) + 0.5 * radial_distance * np.sin(2*angle) + noise_level * 1.2 * np.random.randn(n_samples)
    
    # 產物濃度：交互項
    product_conc = 1.0 + 0.02 * conc_a * conc_b + noise_level * 0.4 * np.random.randn(n_samples)
    
    # ========================================
    # 組織為 DataFrame
    # ========================================
    mode_labels = ['Mode A (Bimodal)'] * n_mode_a + ['Mode B (Middle)'] * n_mode_b
    
    df = pd.DataFrame({
        'Temperature (°C)': temperature,
        'Pressure (bar)': pressure,
        'Feed Flow Rate (kg/h)': feed_flow,
        'Concentration A (mol/L)': conc_a,
        'Concentration B (mol/L)': conc_b,
        'Product Concentration (mol/L)': product_conc,
        'Mode': mode_labels
    })
    
    return df

# 生成訓練和測試數據（測試集使用不同隨機種子，但分佈與訓練集相同）
df_train = generate_nonlinear_reactor_data(n_samples=400, noise_level=0.15, random_seed=42)
df_test = generate_nonlinear_reactor_data(n_samples=100, noise_level=0.15, random_seed=123)

print("✓ 訓練數據生成完成")
print(f"  - 樣本數: {len(df_train)}")
print(f"  - 特徵數: {len(df_train.columns) - 1}")
print(f"  - 模式 A 樣本數: {(df_train['Mode'] == 'Mode A (Bimodal)').sum()}")
print(f"  - 模式 B 樣本數: {(df_train['Mode'] == 'Mode B (Middle)').sum()}")

print("\n✓ 測試數據生成完成")
print(f"  - 樣本數: {len(df_test)}")

print("\n數據結構說明：")
print("  - 模式 A：雙峰分佈（內環 + 外環），代表兩種極端操作條件")
print("  - 模式 B：單峰分佈（中環），代表中等操作條件")
print("  - 模式 B 被模式 A 夾在中間，兩者無法以單一超平面分開（線性不可分）\n")

print("訓練數據概覽:")
display(df_train.head(10))
print("\n數據統計:")
display(df_train.describe())

✓ 訓練數據生成完成
  - 樣本數: 400
  - 特徵數: 6
  - 模式 A 樣本數: 200
  - 模式 B 樣本數: 200

✓ 測試數據生成完成
  - 樣本數: 100

數據結構說明：
  - 模式 A：雙峰分佈（內環 + 外環），代表兩種極端操作條件
  - 模式 B：單峰分佈（中環），代表中等操作條件
  - 模式 B 被模式 A 夾在中間，兩者無法以單一超平面分開（線性不可分）

訓練數據概覽:


,Temperature (°C),Pressure (bar),Feed Flow Rate (kg/h),Concentration A (mol/L),Concentration B (mol/L),Product Concentration (mol/L),Mode
0,352.422296,7.438016,1089.891770,9.765596,7.396727,2.455699,Mode A (Bimodal)
1,348.941005,7.201026,1107.889397,10.218440,7.573469,2.709363,Mode A (Bimodal)
2,358.681856,7.454397,1087.964793,10.099914,7.925115,2.621848,Mode A (Bimodal)
3,361.953164,8.121976,1062.930475,10.050578,7.983314,2.544495,Mode A (Bimodal)
4,354.585552,7.290004,1096.794232,9.916225,8.187227,2.618000,Mode A (Bimodal)
5,358.172070,7.279478,1095.816514,10.595653,8.372369,2.667640,Mode A (Bimodal)
6,372.684739,8.468492,1069.187162,11.127327,8.272488,2.836178,Mode A (Bimodal)
7,365.454764,7.975878,1079.234631,10.279415,8.422724,2.681630,Mode A (Bimodal)
8,363.261444,7.265204,1094.654364,10.641944,8.375726,2.837604,Mode A (Bimodal)
9,364.297455,8.427906,1064.478907,10.584228,7.962084,2.652478,Mode A (Bimodal)



數據統計:


,Temperature (°C),Pressure (bar),Feed Flow Rate (kg/h),Concentration A (mol/L),Concentration B (mol/L),Product Concentration (mol/L)
count,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000
mean,373.360128,14.032023,1139.532348,16.058372,6.047627,2.942697
std,26.591591,5.684491,52.128418,4.598720,4.580392,1.991607
min,332.324367,6.433364,1046.688459,9.397244,-5.686609,-1.655490
25%,352.771297,8.850632,1100.432902,13.063025,3.531846,1.842527
50%,367.528186,13.160660,1135.936894,15.324787,6.086626,2.534728
75%,391.612900,18.477086,1173.837644,18.147748,8.623150,3.781576
max,441.234436,27.813467,1273.269343,25.490870,18.093099,9.648605


### 2.1 數據視覺化探索

In [4]:
# 視覺化原始數據
fig = plt.figure(figsize=(18, 5))

# 子圖1：Temperature vs Pressure
ax1 = plt.subplot(1, 3, 1)
for mode in df_train['Mode'].unique():
    mask = df_train['Mode'] == mode
    ax1.scatter(df_train.loc[mask, 'Temperature (°C)'], 
               df_train.loc[mask, 'Pressure (bar)'],
               label=mode, alpha=0.6, s=40)
ax1.set_xlabel('Temperature (°C)', fontsize=12)
ax1.set_ylabel('Pressure (bar)', fontsize=12)
ax1.set_title('Original Data: Temperature vs Pressure\n(Both Modes Intermixed)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# 子圖2：Feed Flow vs Concentration A
ax2 = plt.subplot(1, 3, 2)
for mode in df_train['Mode'].unique():
    mask = df_train['Mode'] == mode
    ax2.scatter(df_train.loc[mask, 'Feed Flow Rate (kg/h)'], 
               df_train.loc[mask, 'Concentration A (mol/L)'],
               label=mode, alpha=0.6, s=40)
ax2.set_xlabel('Feed Flow Rate (kg/h)', fontsize=12)
ax2.set_ylabel('Concentration A (mol/L)', fontsize=12)
ax2.set_title('Original Data: Feed Flow vs Conc A\n(Mode B Sandwiched by Mode A)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# 子圖3：Concentration A vs B
ax3 = plt.subplot(1, 3, 3)
for mode in df_train['Mode'].unique():
    mask = df_train['Mode'] == mode
    ax3.scatter(df_train.loc[mask, 'Concentration A (mol/L)'], 
               df_train.loc[mask, 'Concentration B (mol/L)'],
               label=mode, alpha=0.6, s=40)
ax3.set_xlabel('Concentration A (mol/L)', fontsize=12)
ax3.set_ylabel('Concentration B (mol/L)', fontsize=12)
ax3.set_title('Original Data: Conc A vs Conc B\n(Three Bands Along Conc A)', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '01_original_data.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ 模式 A (Bimodal)：分佈在內環 (r≈2) 和外環 (r≈5)")
print("✓ 模式 B (Middle)：分佈在中環 (r≈3.5)")
print("✓ 關鍵特性：")
print("   - 濃度 A 是徑向距離的函數：模式 A 位於約 10 與 23 mol/L 兩端，模式 B 位於約 15 mol/L")
print("   - 模式 B 被模式 A 夾在中間，沒有單一線性方向可將 A 與 B 完整分開（線性不可分）")
print("   - 溫度-壓力平面上兩模式相互交錯，難以直接以線性邊界區分")

✓ 模式 A (Bimodal)：分佈在內環 (r≈2) 和外環 (r≈5)
✓ 模式 B (Middle)：分佈在中環 (r≈3.5)
✓ 關鍵特性：
   - 濃度 A 是徑向距離的函數：模式 A 位於約 10 與 23 mol/L 兩端，模式 B 位於約 15 mol/L
   - 模式 B 被模式 A 夾在中間，沒有單一線性方向可將 A 與 B 完整分開（線性不可分）
   - 溫度-壓力平面上兩模式相互交錯，難以直接以線性邊界區分


---
## 3. 數據準備與標準化

In [5]:
# 分離特徵與標籤
feature_cols = ['Temperature (°C)', 'Pressure (bar)', 'Feed Flow Rate (kg/h)', 
                'Concentration A (mol/L)', 'Concentration B (mol/L)', 
                'Product Concentration (mol/L)']

X_train = df_train[feature_cols].values
X_test = df_test[feature_cols].values
y_train = df_train['Mode'].values
y_test = df_test['Mode'].values

print("訓練集特徵形狀:", X_train.shape)
print("測試集特徵形狀:", X_test.shape)

# 數據標準化（重要！）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ 數據標準化完成")
print(f"標準化後均值: {X_train_scaled.mean(axis=0).round(4)}")
print(f"標準化後標準差: {X_train_scaled.std(axis=0).round(4)}")

# ========================================
# 輔助函數：Kernel PCA 解釋變異數與分離指標
# ========================================
def kpca_explained_variance(X, kernel, **kernel_params):
    """以中心化核矩陣的全部正特徵值計算解釋變異數比例。
    KernelPCA(n_components=k) 的 eigenvalues_ 只保留前 k 個，
    直接正規化會得到 100% 的假象，因此需另外計算全部特徵值。
    非正定核（如 sigmoid）的負特徵值予以忽略。"""
    K = pairwise_kernels(X, metric=kernel, filter_params=True, **kernel_params)
    eig = np.linalg.eigvalsh(KernelCenterer().fit_transform(K))[::-1]
    eig = eig[eig > 0]
    return eig / eig.sum()

def separation_metrics(Z, labels, k=5, cv=5):
    """回傳 silhouette 係數與 kNN 交叉驗證準確率（越高代表兩類在降維空間越易分離）。"""
    sil = silhouette_score(Z, labels)
    acc = cross_val_score(KNeighborsClassifier(n_neighbors=k), Z, labels, cv=cv).mean()
    return sil, acc

訓練集特徵形狀: (400, 6)
測試集特徵形狀: (100, 6)

✓ 數據標準化完成
標準化後均值: [-0. -0. -0.  0.  0. -0.]
標準化後標準差: [1. 1. 1. 1. 1. 1.]


---
## 4. 基準模型：傳統 PCA

首先建立傳統 PCA 模型作為基準，觀察線性降維的效果。

**重要提醒**：PCA 的「解釋變異數」和「類別可分性」是兩個不同的概念：
- **解釋變異數**：衡量主成分捕捉了多少原始數據的變異
- **類別可分性**：衡量不同類別在降維空間中是否能被區分開來

⚠️ **高解釋變異數不保證好的分離效果！** 本節同時計算 silhouette 係數與 kNN 交叉驗證準確率，以定量評估分離程度。

In [6]:
# 建立傳統 PCA 模型
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# 檢視解釋變異數
print("傳統 PCA 解釋變異數比例:")
for i, var in enumerate(pca.explained_variance_ratio_, 1):
    print(f"  PC{i}: {var:.4f} ({var*100:.2f}%)")
print(f"\n累積解釋變異數: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.2f}%)")

# 定量分離指標
sil_pca, acc_pca = separation_metrics(X_train_pca, y_train)
print(f"\nPCA 分離指標: silhouette = {sil_pca:.3f}, kNN CV accuracy = {acc_pca:.3f}")

# 視覺化 PCA 結果
fig, ax = plt.subplots(figsize=(8, 6))

for mode in np.unique(y_train):
    mask = y_train == mode
    ax.scatter(X_train_pca[mask, 0], X_train_pca[mask, 1], 
              label=mode, alpha=0.6, s=50)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
ax.set_title('Traditional PCA Result\n(Linear Dimensionality Reduction)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '02_traditional_pca_result.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n【關鍵觀察】傳統 PCA 的限制：")
print("  ✓ 前兩個主成分解釋了約 86.7% 的變異（第 3 個主成分僅約 9.2%）")
print("  ✓ 模式 B 在 PC 平面上呈帶狀，與模式 A 的大部分區域可分開")
print("  ✗ 模式 A 分佈在模式 B 的上方（含右側延伸）與下方，B 呈 V 形帶夾在其間，僅靠單一直線無法分開 A 與 B")
print("  ✗ 模式 A 下方群與模式 B 在 PC1 約 -0.5 ~ 0.5、PC2 約 -1 附近邊界緊鄰，有局部混合")
print("\n重要概念：「高解釋變異數」≠「類別線性可分」")

傳統 PCA 解釋變異數比例:
  PC1: 0.4463 (44.63%)
  PC2: 0.4207 (42.07%)

累積解釋變異數: 0.8670 (86.70%)

PCA 分離指標: silhouette = 0.127, kNN CV accuracy = 0.920



【關鍵觀察】傳統 PCA 的限制：
  ✓ 前兩個主成分解釋了約 86.7% 的變異（第 3 個主成分僅約 9.2%）
  ✓ 模式 B 在 PC 平面上呈帶狀，與模式 A 的大部分區域可分開
  ✗ 模式 A 分佈在模式 B 的上方（含右側延伸）與下方，B 呈 V 形帶夾在其間，僅靠單一直線無法分開 A 與 B
  ✗ 模式 A 下方群與模式 B 在 PC1 約 -0.5 ~ 0.5、PC2 約 -1 附近邊界緊鄰，有局部混合

重要概念：「高解釋變異數」≠「類別線性可分」


---
## 5. Kernel PCA 模型建立

### 5.1 使用 RBF 核函數的 Kernel PCA

In [7]:
# 建立 Kernel PCA 模型（RBF 核，gamma=0.1）
# scikit-learn 預設 gamma=None 即 1/n_features；此處明確指定 gamma=0.1
kpca_rbf = KernelPCA(n_components=2, kernel='rbf', gamma=0.1, fit_inverse_transform=True, random_state=SEED)
X_train_kpca_rbf = kpca_rbf.fit_transform(X_train_scaled)
X_test_kpca_rbf = kpca_rbf.transform(X_test_scaled)

# 解釋變異數：必須以「全部」特徵值正規化（核空間中的變異）
explained_var_ratio = kpca_explained_variance(X_train_scaled, kernel='rbf', gamma=0.1)
n_90 = int(np.searchsorted(np.cumsum(explained_var_ratio), 0.90) + 1)

print("Kernel PCA (RBF, gamma=0.1) 解釋變異數比例（核空間）:")
for i, var in enumerate(explained_var_ratio[:5], 1):
    print(f"  KPC{i}: {var:.4f} ({var*100:.2f}%)")
print(f"\n前 2 個核主成分累積解釋變異數: {explained_var_ratio[:2].sum():.4f} ({explained_var_ratio[:2].sum()*100:.2f}%)")
print(f"累積解釋變異數達 90% 需要的核主成分數: {n_90}")

# 定量分離指標
sil_kpca, acc_kpca = separation_metrics(X_train_kpca_rbf, y_train)
print(f"\nKernel PCA (RBF, gamma=0.1) 分離指標: silhouette = {sil_kpca:.3f}, kNN CV accuracy = {acc_kpca:.3f}")
print(f"傳統 PCA 分離指標                 : silhouette = {sil_pca:.3f}, kNN CV accuracy = {acc_pca:.3f}")

# 視覺化 Kernel PCA (RBF) 結果
fig, ax = plt.subplots(figsize=(8, 6))

for mode in np.unique(y_train):
    mask = y_train == mode
    ax.scatter(X_train_kpca_rbf[mask, 0], X_train_kpca_rbf[mask, 1], 
              label=mode, alpha=0.6, s=50)

ax.set_xlabel(f'KPC1 ({explained_var_ratio[0]*100:.2f}%)')
ax.set_ylabel(f'KPC2 ({explained_var_ratio[1]*100:.2f}%)')
ax.set_title('Kernel PCA with RBF Kernel (gamma=0.1)\n(Non-linear Dimensionality Reduction)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '03_kernel_pca_rbf_result.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Kernel PCA (RBF, gamma=0.1) 觀察：")
print("  - 模式 A 分佈在外圍（左側、右側與右上方），模式 B 位於內側的弧形帶，呈現「外-內」層次")
print("  - 前兩個核主成分只解釋約 47% 的核空間變異，A、B 在右上方邊界仍有少量混合")
print("  - gamma=0.1 時 silhouette 略低於 PCA、kNN 準確率略高於 PCA，差異不大，需調整 gamma（見 5.3 節）")

Kernel PCA (RBF, gamma=0.1) 解釋變異數比例（核空間）:
  KPC1: 0.2651 (26.51%)
  KPC2: 0.2068 (20.68%)
  KPC3: 0.1259 (12.59%)
  KPC4: 0.0759 (7.59%)
  KPC5: 0.0723 (7.23%)

前 2 個核主成分累積解釋變異數: 0.4719 (47.19%)
累積解釋變異數達 90% 需要的核主成分數: 10

Kernel PCA (RBF, gamma=0.1) 分離指標: silhouette = 0.102, kNN CV accuracy = 0.943
傳統 PCA 分離指標                 : silhouette = 0.127, kNN CV accuracy = 0.920



✓ Kernel PCA (RBF, gamma=0.1) 觀察：
  - 模式 A 分佈在外圍（左側、右側與右上方），模式 B 位於內側的弧形帶，呈現「外-內」層次
  - 前兩個核主成分只解釋約 47% 的核空間變異，A、B 在右上方邊界仍有少量混合
  - gamma=0.1 時 silhouette 略低於 PCA、kNN 準確率略高於 PCA，差異不大，需調整 gamma（見 5.3 節）


### 5.2 不同核函數的比較

比較不同核函數（Linear, Poly, RBF, Sigmoid）的效果

In [8]:
# 定義不同核函數
kernels = {
    'Linear': {'kernel': 'linear'},
    'Polynomial (d=2)': {'kernel': 'poly', 'degree': 2, 'gamma': 0.1},
    'Polynomial (d=3)': {'kernel': 'poly', 'degree': 3, 'gamma': 0.1},
    'RBF (gamma=0.1)': {'kernel': 'rbf', 'gamma': 0.1},
    'RBF (gamma=1)': {'kernel': 'rbf', 'gamma': 1},
    'Sigmoid': {'kernel': 'sigmoid', 'gamma': 0.1}
}

# 建立模型並轉換數據
results = {}
for name, params in kernels.items():
    kpca = KernelPCA(n_components=2, **params, fit_inverse_transform=True, random_state=SEED)
    X_transformed = kpca.fit_transform(X_train_scaled)
    evr = kpca_explained_variance(X_train_scaled, **params)
    sil, acc = separation_metrics(X_transformed, y_train)
    results[name] = {
        'model': kpca,
        'X_transformed': X_transformed,
        'explained_var': evr[:2],
        'silhouette': sil,
        'knn_acc': acc
    }

# 視覺化比較
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (name, result) in enumerate(results.items()):
    X_transformed = result['X_transformed']
    explained_var = result['explained_var']
    
    for mode in np.unique(y_train):
        mask = y_train == mode
        axes[idx].scatter(X_transformed[mask, 0], X_transformed[mask, 1], 
                         label=mode, alpha=0.6, s=40)
    
    axes[idx].set_xlabel(f'Component 1 ({explained_var[0]*100:.1f}%)')
    axes[idx].set_ylabel(f'Component 2 ({explained_var[1]*100:.1f}%)')
    axes[idx].set_title(f'{name}\nTop-2: {explained_var.sum()*100:.1f}% | Sil={result["silhouette"]:.2f}, kNN={result["knn_acc"]:.2f}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '04_kernel_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# 定量比較表
summary_kernel = pd.DataFrame({
    'Top-2 EVR (%)': {k: v['explained_var'].sum() * 100 for k, v in results.items()},
    'Silhouette': {k: v['silhouette'] for k, v in results.items()},
    'kNN CV Acc': {k: v['knn_acc'] for k, v in results.items()},
}).round(3)
summary_kernel.loc['Traditional PCA'] = [pca.explained_variance_ratio_.sum() * 100, sil_pca, acc_pca]
display(summary_kernel.round(3))

,Top-2 EVR (%),Silhouette,kNN CV Acc
Linear,86.696,0.127,0.920
Polynomial (d=2),70.839,0.143,0.922
Polynomial (d=3),65.566,0.153,0.922
RBF (gamma=0.1),47.188,0.102,0.943
RBF (gamma=1),15.399,0.260,0.980
Sigmoid,84.098,0.114,0.895
Traditional PCA,86.696,0.127,0.920


**結果觀察**：
- **Linear**：與傳統 PCA 完全相同（前兩成分 86.7%）
- **Polynomial (d=2, d=3)**：silhouette 略升至 0.14–0.15，kNN 準確率幾乎不變（0.92），改善有限
- **RBF (gamma=0.1)**：模式 A 呈馬蹄形分佈在外圍，模式 B 位於內側，視覺上有層次，但定量指標與 PCA 差異不大
- **RBF (gamma=1)**：silhouette 0.26、kNN 準確率 0.98，是六種設定中分離指標最好者；模式 A 集中在右側，模式 B 位於左側，仍有一小群 A 嵌入 B 區域
- **Sigmoid**：非正定核，kNN 準確率最低（0.895），不適合本數據
- 前兩個成分的解釋變異數隨 $\gamma$ 增大而下降（RBF gamma=1 僅 15.4%），解釋變異數**不能**作為核函數比較依據

### 5.3 RBF 核的 gamma 參數調整

gamma 參數控制核函數的寬度，影響非線性程度：
- gamma 大：局部性強，可能過擬合
- gamma 小：全局性強，可能欠擬合

In [9]:
# 測試不同 gamma 值
gamma_values = [0.001, 0.01, 0.1, 0.5, 1, 2]
gamma_rows = []

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, gamma in enumerate(gamma_values):
    kpca = KernelPCA(n_components=2, kernel='rbf', gamma=gamma, 
                     fit_inverse_transform=True, random_state=SEED)
    X_transformed = kpca.fit_transform(X_train_scaled)
    evr = kpca_explained_variance(X_train_scaled, kernel='rbf', gamma=gamma)
    sil, acc = separation_metrics(X_transformed, y_train)
    gamma_rows.append({'gamma': gamma, 'Top-2 EVR (%)': evr[:2].sum() * 100,
                       'Silhouette': sil, 'kNN CV Acc': acc})
    
    for mode in np.unique(y_train):
        mask = y_train == mode
        axes[idx].scatter(X_transformed[mask, 0], X_transformed[mask, 1], 
                         label=mode, alpha=0.6, s=40, edgecolors='white', linewidth=0.5)
    
    axes[idx].set_xlabel(f'KPC1 ({evr[0]*100:.1f}%)')
    axes[idx].set_ylabel(f'KPC2 ({evr[1]*100:.1f}%)')
    axes[idx].set_title(f'RBF Kernel: gamma = {gamma}\nTop-2: {evr[:2].sum()*100:.1f}% | Sil={sil:.2f}, kNN={acc:.2f}')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '05_gamma_tuning.png', dpi=300, bbox_inches='tight')
plt.show()

df_gamma = pd.DataFrame(gamma_rows).set_index('gamma').round(3)
display(df_gamma)

,Top-2 EVR (%),Silhouette,kNN CV Acc
gamma,,,
0.001,85.946,0.127,0.922
0.010,79.720,0.125,0.930
0.100,47.188,0.102,0.943
0.500,22.898,0.230,0.938
1.000,15.399,0.260,0.980
2.000,9.847,0.281,0.982


**結果觀察**：
- **gamma = 0.001**：核函數過寬，結果與線性 PCA 幾乎相同（silhouette 0.127、前兩成分 85.9%）
- **gamma = 0.01**：開始出現非線性效果，但 A、B 仍大量交錯
- **gamma = 0.1**：模式 A 呈馬蹄形分佈在左、右與右上外圍，模式 B 位於內側（上方弧形與下方群集），kNN 準確率 0.943，但 A、B 在右上方交會
- **gamma = 0.5 – 2**：模式 A 集中在右側、模式 B 集中在左側，silhouette 由 0.23 升至 0.28，kNN 準確率 0.94–0.98；但仍有一小群模式 A 嵌入模式 B 所在區域，並非「完美分離」
- gamma 增大時，前兩成分解釋變異數由 85.9% 降至 9.8%，高 gamma 下變異分散到許多成分，需搭配後續任務的指標選擇 gamma
- 本數據在掃描範圍內以 gamma = 1 – 2 分離指標最佳；真實應用應以交叉驗證（針對後續分類、迴歸或監控任務）決定 gamma，而不是只看 2D 圖

---
## 6. 模型評估與比較

### 6.1 PCA vs Kernel PCA 的視覺化比較

In [10]:
# 並排比較 PCA 和 Kernel PCA 的效果
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：傳統 PCA
for mode in np.unique(y_train):
    mask = y_train == mode
    axes[0].scatter(X_train_pca[mask, 0], X_train_pca[mask, 1], 
                   label=mode, alpha=0.6, s=60)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
axes[0].set_title(f'Traditional PCA (Linear)\nSil={sil_pca:.2f}, kNN={acc_pca:.2f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右圖：Kernel PCA (RBF, gamma=0.1)
for mode in np.unique(y_train):
    mask = y_train == mode
    axes[1].scatter(X_train_kpca_rbf[mask, 0], X_train_kpca_rbf[mask, 1], 
                   label=mode, alpha=0.6, s=60)
axes[1].set_xlabel(f'KPC1 ({explained_var_ratio[0]*100:.2f}%)')
axes[1].set_ylabel(f'KPC2 ({explained_var_ratio[1]*100:.2f}%)')
axes[1].set_title(f'Kernel PCA with RBF (gamma=0.1)\nSil={sil_kpca:.2f}, kNN={acc_kpca:.2f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '06_pca_vs_kernel_pca.png', dpi=300, bbox_inches='tight')
plt.show()

# 以最佳 gamma（依 kNN 準確率與 silhouette）補充比較
best_gamma = df_gamma['kNN CV Acc'].idxmax()
print("【定量比較】")
print(f"  傳統 PCA              : Top-2 EVR = {pca.explained_variance_ratio_.sum()*100:.1f}%, silhouette = {sil_pca:.3f}, kNN CV acc = {acc_pca:.3f}")
print(f"  Kernel PCA (gamma=0.1): Top-2 EVR = {explained_var_ratio[:2].sum()*100:.1f}%, silhouette = {sil_kpca:.3f}, kNN CV acc = {acc_kpca:.3f}")
print(f"  Kernel PCA (gamma={best_gamma}) : Top-2 EVR = {df_gamma.loc[best_gamma, 'Top-2 EVR (%)']:.1f}%, silhouette = {df_gamma.loc[best_gamma, 'Silhouette']:.3f}, kNN CV acc = {df_gamma.loc[best_gamma, 'kNN CV Acc']:.3f}")
print("\n【解讀】")
print("  - 兩種方法的解釋變異數不可直接比較：PCA 為原始空間變異，Kernel PCA 為核空間變異")
print("  - gamma=0.1 時 Kernel PCA 與 PCA 的分離效果差異不大；適當增大 gamma 後分離指標才明顯提升")
print("  - 評估降維效果不能只看解釋變異數，需搭配視覺化與分離指標")

【定量比較】
  傳統 PCA              : Top-2 EVR = 86.7%, silhouette = 0.127, kNN CV acc = 0.920
  Kernel PCA (gamma=0.1): Top-2 EVR = 47.2%, silhouette = 0.102, kNN CV acc = 0.943
  Kernel PCA (gamma=2.0) : Top-2 EVR = 9.8%, silhouette = 0.281, kNN CV acc = 0.982

【解讀】
  - 兩種方法的解釋變異數不可直接比較：PCA 為原始空間變異，Kernel PCA 為核空間變異
  - gamma=0.1 時 Kernel PCA 與 PCA 的分離效果差異不大；適當增大 gamma 後分離指標才明顯提升
  - 評估降維效果不能只看解釋變異數，需搭配視覺化與分離指標


### 6.2 測試集驗證

使用訓練好的 scaler 與 Kernel PCA 轉換測試集，並以訓練集上的 kNN 分類器評估測試集準確率。

**注意**：測試集與訓練集來自相同的數據生成器（僅隨機種子不同），因此此驗證只能說明模型在相同分佈下的泛化能力，不能代表對未見過的操作條件或故障的偵測能力。

In [11]:
# 在測試集上應用 Kernel PCA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：訓練集
for mode in np.unique(y_train):
    mask = y_train == mode
    axes[0].scatter(X_train_kpca_rbf[mask, 0], X_train_kpca_rbf[mask, 1], 
                   label=mode, alpha=0.6, s=60)
axes[0].set_xlabel(f'KPC1 ({explained_var_ratio[0]*100:.2f}%)')
axes[0].set_ylabel(f'KPC2 ({explained_var_ratio[1]*100:.2f}%)')
axes[0].set_title('Training Set - Kernel PCA (RBF, gamma=0.1)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 右圖：測試集
for mode in np.unique(y_test):
    mask = y_test == mode
    axes[1].scatter(X_test_kpca_rbf[mask, 0], X_test_kpca_rbf[mask, 1], 
                   label=mode, alpha=0.6, s=60)
axes[1].set_xlabel(f'KPC1 ({explained_var_ratio[0]*100:.2f}%)')
axes[1].set_ylabel(f'KPC2 ({explained_var_ratio[1]*100:.2f}%)')
axes[1].set_title('Test Set - Kernel PCA (RBF, gamma=0.1)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / '07_test_set_validation.png', dpi=300, bbox_inches='tight')
plt.show()

# 測試集 kNN 準確率（分類器僅用訓練集擬合）
knn_pca = KNeighborsClassifier(n_neighbors=5).fit(X_train_pca, y_train)
knn_kpca = KNeighborsClassifier(n_neighbors=5).fit(X_train_kpca_rbf, y_train)
print(f"測試集 kNN 準確率 - PCA: {knn_pca.score(X_test_pca, y_test):.3f}")
print(f"測試集 kNN 準確率 - Kernel PCA (RBF, gamma=0.1): {knn_kpca.score(X_test_kpca_rbf, y_test):.3f}")

print("\n✓ 測試集驗證結果：")
print("  - 測試集點的分佈與訓練集相似（A 在左右兩側，B 在中間弧形帶）")
print("  - 圖中軸標籤的百分比為訓練集的核空間解釋變異數")
print("  - 測試集僅 100 筆且與訓練集同分佈，兩種方法的準確率差異不具統計顯著性")

測試集 kNN 準確率 - PCA: 0.980
測試集 kNN 準確率 - Kernel PCA (RBF, gamma=0.1): 0.990

✓ 測試集驗證結果：
  - 測試集點的分佈與訓練集相似（A 在左右兩側，B 在中間弧形帶）
  - 圖中軸標籤的百分比為訓練集的核空間解釋變異數
  - 測試集僅 100 筆且與訓練集同分佈，兩種方法的準確率差異不具統計顯著性


---
## 7. 模型保存與載入

In [12]:
# 保存模型
joblib.dump(scaler, MODEL_DIR / 'scaler.pkl')
joblib.dump(pca, MODEL_DIR / 'pca_model.pkl')
joblib.dump(kpca_rbf, MODEL_DIR / 'kernel_pca_rbf_model.pkl')

print("✓ 模型已保存至:", MODEL_DIR)
print("  - scaler.pkl")
print("  - pca_model.pkl")
print("  - kernel_pca_rbf_model.pkl")

# 載入模型示例
loaded_scaler = joblib.load(MODEL_DIR / 'scaler.pkl')
loaded_kpca = joblib.load(MODEL_DIR / 'kernel_pca_rbf_model.pkl')

# 使用載入的模型進行轉換
X_test_transformed = loaded_kpca.transform(loaded_scaler.transform(X_test))

print("\n✓ 模型載入測試成功")
print(f"轉換後數據形狀: {X_test_transformed.shape}")

✓ 模型已保存至: D:\MyGit\ChemE-3590\Part_2\Unit06\outputs\P2_Unit06_Kernel_PCA\models
  - scaler.pkl
  - pca_model.pkl
  - kernel_pca_rbf_model.pkl

✓ 模型載入測試成功
轉換後數據形狀: (100, 2)


---
## 8. 實務應用：製程監控統計量建立

使用 Kernel PCA 建立非線性製程監控統計量 **Hotelling's T²** 與 **SPE (Squared Prediction Error)**。

**T² 統計量**（監控核主成分空間內的變異）：

$$
T^2 = \sum_{i=1}^{k} \frac{z_i^2}{s_i^2}, \quad s_i^2 = \frac{\mu_i}{n}
$$

其中 $z_i$ 為第 $i$ 個核主成分得分， $\mu_i$ 為中心化核矩陣 $\tilde{\mathbf{K}}$ 的特徵值， $n$ 為訓練樣本數。scikit-learn 的 `transform` 輸出滿足 $\sum_j z_{ji}^2 = \mu_i$ ，所以訓練得分的變異數為 $\mu_i / n$ ，不是 $\mu_i$ 。若錯把 $\mu_i$ 當作得分變異數，T² 會被縮小 $n$ 倍。

**SPE 統計量**（監控核主成分空間以外的殘差）：

$$
\mathrm{SPE} = \tilde{k}(\mathbf{x}, \mathbf{x}) - \sum_{i=1}^{k} z_i^2
$$

其中 $\tilde{k}(\mathbf{x}, \mathbf{x})$ 是中心化後的核函數自我內積。RBF 核下，遠離訓練數據的樣本其核函數值趨近 0，得分 $z_i$ 也趨近 0，因此 T² 可能無法偵測遠離正常範圍的故障，此時 SPE 才能發揮作用。

**控制限**：以正常訓練數據統計量的 99% 百分位數作為經驗控制限。

In [13]:
GAMMA_MON = 0.1
K_train = rbf_kernel(X_train_scaled, X_train_scaled, gamma=GAMMA_MON)
n_train = X_train_scaled.shape[0]

def compute_t2(X_scaled):
    """Hotelling's T²：得分除以訓練得分變異數 (mu/n，mu 為中心化核矩陣特徵值)。"""
    scores = kpca_rbf.transform(X_scaled)
    score_var = kpca_rbf.eigenvalues_ / n_train
    return np.sum(scores**2 / score_var, axis=1)

def compute_spe(X_scaled):
    """SPE：中心化核自我內積減去前 k 個核主成分得分平方和。"""
    K_new = rbf_kernel(X_scaled, X_train_scaled, gamma=GAMMA_MON)
    k_self_centered = 1.0 - 2.0 * K_new.mean(axis=1) + K_train.mean()  # RBF: k(x, x) = 1
    scores = kpca_rbf.transform(X_scaled)
    return np.maximum(k_self_centered - np.sum(scores**2, axis=1), 0.0)

# 訓練集統計量與 99% 經驗控制限
t2_train, spe_train = compute_t2(X_train_scaled), compute_spe(X_train_scaled)
t2_limit, spe_limit = np.percentile(t2_train, 99), np.percentile(spe_train, 99)

# 正常測試集（與訓練集同分佈）
t2_test, spe_test = compute_t2(X_test_scaled), compute_spe(X_test_scaled)

# 模擬故障：溫度感測器偏移 +3 個標準差（標準化空間）
X_fault_scaled = X_test_scaled.copy()
X_fault_scaled[:, 0] += 3.0
t2_fault, spe_fault = compute_t2(X_fault_scaled), compute_spe(X_fault_scaled)

# 組成「前 100 筆正常 + 後 100 筆故障」的監控序列（隨機打亂各段順序，避免結構性排序）
rng = np.random.default_rng(SEED)
idx_normal, idx_fault = rng.permutation(len(X_test_scaled)), rng.permutation(len(X_test_scaled))
t2_seq = np.concatenate([t2_test[idx_normal], t2_fault[idx_fault]])
spe_seq = np.concatenate([spe_test[idx_normal], spe_fault[idx_fault]])
fault_start = len(X_test_scaled)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, stat, limit, name in [(axes[0], t2_seq, t2_limit, "Hotelling's T²"),
                              (axes[1], spe_seq, spe_limit, 'SPE')]:
    ax.plot(np.arange(len(stat)), stat, 'o', alpha=0.6, markersize=4)
    ax.axhline(limit, color='r', linestyle='--', linewidth=2, label=f'99% Control Limit = {limit:.3f}')
    ax.axvline(fault_start - 0.5, color='k', linestyle=':', label='Fault starts (temperature +3 sigma)')
    ax.set_ylabel(name)
    ax.set_title(f'Kernel PCA - {name} Control Chart')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
axes[1].set_xlabel('Sample Index')

plt.tight_layout()
plt.savefig(FIG_DIR / '08_t2_spe_control_chart.png', dpi=300, bbox_inches='tight')
plt.show()

def alarm_rate(stat, limit):
    return np.mean(stat > limit) * 100

print("✓ T² 與 SPE 統計量建立完成")
print(f"  - T² 99% 控制限: {t2_limit:.4f} (訓練集 T² 平均值: {t2_train.mean():.3f}，理論上約等於主成分數 2)")
print(f"  - SPE 99% 控制限: {spe_limit:.4f}")
print("\n誤報率（正常數據超出控制限的比例）：")
print(f"  - 訓練集  T²: {alarm_rate(t2_train, t2_limit):.2f}%  SPE: {alarm_rate(spe_train, spe_limit):.2f}%  （由百分位數定義，必然約為 1%）")
print(f"  - 測試集  T²: {alarm_rate(t2_test, t2_limit):.2f}%  SPE: {alarm_rate(spe_test, spe_limit):.2f}%  （獨立驗證）")
print("\n故障偵測率（溫度 +3 sigma 偏移）：")
print(f"  - T²  : {alarm_rate(t2_fault, t2_limit):.2f}%")
print(f"  - SPE : {alarm_rate(spe_fault, spe_limit):.2f}%")

✓ T² 與 SPE 統計量建立完成
  - T² 99% 控制限: 3.9127 (訓練集 T² 平均值: 2.000，理論上約等於主成分數 2)
  - SPE 99% 控制限: 0.8291

誤報率（正常數據超出控制限的比例）：
  - 訓練集  T²: 1.00%  SPE: 1.00%  （由百分位數定義，必然約為 1%）
  - 測試集  T²: 0.00%  SPE: 2.00%  （獨立驗證）

故障偵測率（溫度 +3 sigma 偏移）：
  - T²  : 0.00%
  - SPE : 51.00%


---
## 9. 總結與關鍵要點

### 9.1 Kernel PCA 核心概念回顧

**【重要概念釐清】解釋變異數 vs 類別可分性**

- **傳統 PCA**：前兩個主成分解釋 86.70% 的原始空間變異，但模式 A 與 B 在局部仍有混合（silhouette 0.127、kNN 準確率 0.920）
- **Kernel PCA（RBF, gamma=0.1）**：前兩個核主成分只解釋約 47.19% 的核空間變異，分離指標與 PCA 接近（silhouette 0.102、kNN 準確率 0.943）
- **Kernel PCA（RBF, gamma=1）**：前兩個核主成分只解釋 15.4% 的核空間變異，但 silhouette 0.26、kNN 準確率 0.98，分離效果最好

兩點啟示：
1. PCA 與 Kernel PCA 的解釋變異數分別定義在原始空間與核空間，**不能直接比較**
2. 解釋變異數高不代表類別可分，解釋變異數低也不代表降維失敗；評估降維效果需搭配視覺化與分離指標（silhouette、kNN 準確率），並以後續任務的表現為最終依據

> 註：scikit-learn 的 `KernelPCA(n_components=k).eigenvalues_` 只保留前 $k$ 個特徵值，不能直接用來正規化解釋變異數，必須以全部特徵值計算（見 `kpca_explained_variance()`）。

**核心觀念**：

1. **核技巧 (Kernel Trick)**：
   - 隱式地將數據映射至高維特徵空間
   - 使用核函數計算內積，避免顯式高維計算
   - 能夠捕捉複雜的非線性關係

2. **核函數選擇**：
   - **RBF 核**：最通用，適合大多數非線性情況（建議首選）
   - **多項式核**：適合已知的多項式關係
   - **線性核**：等同於傳統 PCA
   - **Sigmoid 核**：非正定核，本案例分離效果最差，一般不建議
   - gamma 控制非線性程度：過小趨近 PCA，過大則結構過度局部化

3. **優勢與挑戰**：
   - 優勢：捕捉非線性結構、改善降維與視覺化
   - 挑戰：計算複雜度 $O(n^2)$ 記憶體與 $O(n^3)$ 特徵分解、超參數調整、可解釋性較差

### 9.2 化工應用建議

1. **製程監控**：
   - 以 Kernel PCA 的 $T^2$ 與 SPE 統計量監控非線性製程
   - 本案例中，溫度 +3 sigma 偏移時 $T^2$ 偵測率 0%，SPE 偵測率 51%：RBF 核對遠離訓練數據的樣本，其核特徵趨近於零， $T^2$ 反而不會升高，因此**必須搭配 SPE**
   - gamma 建議從 $1/n_{features}$ 或 0.1 開始，以交叉驗證或後續任務指標調整

2. **軟感測器開發**：
   - Kernel PCA 作為非線性特徵提取步驟，降低輸入維度並緩解多重共線性
   - 是否提升預測準確度，需以交叉驗證確認

3. **操作模式識別**：
   - 在降維空間中分離不同操作模式
   - 可結合分群演算法識別模式切換

4. **何時使用 Kernel PCA**：
   - 數據存在明顯非線性結構，且傳統 PCA 效果不佳
   - 樣本數適中（約 10000 以下）
   - 主要用於降維、視覺化、特徵提取、異常偵測

5. **何時使用傳統 PCA**：
   - 數據關係近似線性
   - 樣本數非常大
   - 需要解釋主成分的物理意義（載荷）
   - 需要精確重建原始數據

### 9.3 實務操作流程

1. 數據標準化（Z-score，只用訓練集的平均與標準差）
2. 選擇核函數（建議從 RBF 開始）
3. 調整超參數（gamma、n_components），以分離指標或後續任務的交叉驗證結果判斷
4. 訓練 Kernel PCA 模型
5. 視覺化降維結果並與傳統 PCA 比較
6. 評估分離效果（silhouette、kNN 準確率）
7. 必要時調整超參數並重複步驟 3 至 6
8. 以獨立測試集驗證
9. 建立製程監控統計量（ $T^2$ 、SPE）並以訓練數據的 99% 分位數設定管制界限
10. 儲存模型（scaler、Kernel PCA）供即時應用

### 9.4 延伸學習

- 製程監控進階主題：Multi-way Kernel PCA、動態 Kernel PCA
- 核方法家族：Kernel SVM、Kernel Ridge Regression
- 其他非線性降維方法：t-SNE、UMAP、Isomap、LLE
- 深度學習降維：Autoencoder、VAE